# Feature Engineering 2025H1 (Distributed Spark)

This notebook builds a dense 10-minute demand table by ZoneID and creates time-series features.

Outputs:
- /user/data/feature_engineering/demand_prediction_2025H1_dense_10m
- /user/data/feature_engineering/demand_prediction_2025H1_features

In [ ]:
import json
import urllib.request

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder
    .appName("DemandPredictionFeatureEngineering2025H1")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    print("Spark master:", spark.sparkContext.master)
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")

In [ ]:
RAW_PATH = "/user/data/raw/*.parquet"
OUT_DENSE = "/user/data/feature_engineering/demand_prediction_2025H1_dense_10m"
OUT_FEATURES = "/user/data/feature_engineering/demand_prediction_2025H1_features"

START_TS = "2025-01-01 00:00:00"
END_TS = "2025-07-01 00:00:00"

ZONE_COL = "PULocationID"
BIN_COL = "pickup_bin_10m"
TARGET_COL = "pickup_demand"

raw_df = spark.read.parquet(RAW_PATH)
pickup_col_candidates = ["tpep_pickup_datetime", "lpep_pickup_datetime", "pickup_datetime"]
pickup_col = next((c for c in pickup_col_candidates if c in raw_df.columns), None)

if pickup_col is None:
    raise ValueError(f"Cannot find pickup timestamp column. Tried: {pickup_col_candidates}")
if ZONE_COL not in raw_df.columns:
    raise ValueError(f"Missing column: {ZONE_COL}")

trips = (
    raw_df
    .select(
        F.to_timestamp(F.col(pickup_col)).alias("pickup_ts"),
        F.col(ZONE_COL).cast("int").alias(ZONE_COL)
    )
    .where(F.col("pickup_ts").isNotNull() & F.col(ZONE_COL).isNotNull())
    .where((F.col("pickup_ts") >= F.lit(START_TS)) & (F.col("pickup_ts") < F.lit(END_TS)))
    .where(F.col(ZONE_COL) > 0)
)

print("Trips in 2025H1:", trips.count())
cluster_util("after_raw_filter")

In [ ]:
demand_10m = (
    trips
    .withColumn(BIN_COL, F.to_timestamp(F.from_unixtime(F.floor(F.unix_timestamp("pickup_ts") / 600) * 600)))
    .groupBy(ZONE_COL, BIN_COL)
    .agg(F.count(F.lit(1)).cast("double").alias(TARGET_COL))
)

zone_dim = demand_10m.select(ZONE_COL).distinct()
time_bounds = demand_10m.agg(F.min(BIN_COL).alias("min_ts"), F.max(BIN_COL).alias("max_ts")).first()

if time_bounds["min_ts"] is None or time_bounds["max_ts"] is None:
    raise ValueError("No demand rows found in selected period.")

time_dim = (
    spark.range(1)
    .select(F.sequence(F.lit(time_bounds["min_ts"]), F.lit(time_bounds["max_ts"]), F.expr("interval 10 minutes")).alias("bins"))
    .select(F.explode(F.col("bins")).alias(BIN_COL))
)

dense_df = (
    zone_dim.crossJoin(time_dim)
    .join(demand_10m, on=[ZONE_COL, BIN_COL], how="left")
    .fillna({TARGET_COL: 0.0})
    .repartition(64, ZONE_COL)
    .cache()
)

print("Dense rows:", dense_df.count())
cluster_util("after_dense_table")

In [ ]:
w = Window.partitionBy(ZONE_COL).orderBy(BIN_COL)
w_hist_6 = w.rowsBetween(-6, -1)
w_hist_18 = w.rowsBetween(-18, -1)
w_count = Window.partitionBy(ZONE_COL)

features_df = (
    dense_df
    .withColumn("hour", F.hour(BIN_COL).cast("double"))
    .withColumn("dow", F.dayofweek(BIN_COL).cast("double"))
    .withColumn("month", F.month(BIN_COL).cast("double"))
    .withColumn("is_weekend", F.when(F.dayofweek(BIN_COL).isin([1, 7]), F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn("lag_1", F.lag(TARGET_COL, 1).over(w).cast("double"))
    .withColumn("lag_2", F.lag(TARGET_COL, 2).over(w).cast("double"))
    .withColumn("lag_3", F.lag(TARGET_COL, 3).over(w).cast("double"))
    .withColumn("lag_6", F.lag(TARGET_COL, 6).over(w).cast("double"))
    .withColumn("lag_12", F.lag(TARGET_COL, 12).over(w).cast("double"))
    .withColumn("lag_144", F.lag(TARGET_COL, 144).over(w).cast("double"))
    .withColumn("lag_1008", F.lag(TARGET_COL, 1008).over(w).cast("double"))
    .withColumn("roll_mean_6", F.avg(TARGET_COL).over(w_hist_6).cast("double"))
    .withColumn("roll_mean_18", F.avg(TARGET_COL).over(w_hist_18).cast("double"))
    .withColumn("roll_std_18", F.stddev_pop(TARGET_COL).over(w_hist_18).cast("double"))
    .dropna()
    .withColumn("rn", F.row_number().over(w))
    .withColumn("n_zone", F.count(F.lit(1)).over(w_count))
    .withColumn("train_cut", F.floor(F.col("n_zone") * F.lit(0.7)))
    .withColumn("split", F.when(F.col("rn") <= F.col("train_cut"), F.lit("train")).otherwise(F.lit("test")))
    .cache()
)

print("Feature rows:", features_df.count())
features_df.groupBy("split").count().show()
cluster_util("after_feature_engineering")

In [ ]:
dense_df.write.mode("overwrite").parquet(OUT_DENSE)
features_df.write.mode("overwrite").parquet(OUT_FEATURES)

print("Saved:")
print("-", OUT_DENSE)
print("-", OUT_FEATURES)
cluster_util("after_feature_write")

In [ ]:
zone_rank = (
    dense_df.groupBy(ZONE_COL).agg(F.sum(TARGET_COL).alias("total"))
    .orderBy(F.col("total").desc())
    .limit(3)
)
top_zones = [r[ZONE_COL] for r in zone_rank.collect()]

plot_df = (
    dense_df.where(F.col(ZONE_COL).isin(top_zones))
    .where(F.col(BIN_COL) >= F.to_timestamp(F.lit("2025-06-01 00:00:00")))
    .orderBy(ZONE_COL, BIN_COL)
    .toPandas()
)

if len(plot_df) > 0:
    fig, axes = plt.subplots(len(top_zones), 1, figsize=(16, 4 * len(top_zones)), sharex=True)
    if len(top_zones) == 1:
        axes = [axes]
    for ax, z in zip(axes, top_zones):
        tmp = plot_df[plot_df[ZONE_COL] == z]
        ax.plot(tmp[BIN_COL], tmp[TARGET_COL], linewidth=0.9)
        ax.set_title(f"Zone {z} demand (June 2025)")
        ax.set_ylabel("pickups/10m")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No data for plotting.")